In [2]:
import json
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

class ICDARDataLoader:
    def __init__(self, data_dir, json_path, img_size=(128, 32), max_text_len=50):
        """
        data_dir: путь к папке с изображениями
        json_path: путь к JSON файлу с аннотациями
        img_size: (width, height) для ресайза изображений
        max_text_len: максимальная длина текстовой строки
        """
        self.data_dir = data_dir
        self.json_path = json_path
        self.img_width, self.img_height = img_size
        self.max_text_len = max_text_len
        self.char_to_idx = {}
        self.idx_to_char = {}
        self.num_classes = None
        
    def load_data(self):
        """Загрузка данных из JSON файла"""
        with open(self.json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        images = []
        labels = []
        
        for item in data:
            # Получаем путь к изображению
            img_path = os.path.join(self.data_dir, item['image_path'])
            
            # Проверяем существование файла
            if os.path.exists(img_path):
                # Загружаем и обрабатываем изображение
                img = cv2.imread(img_path)
                if img is not None:
                    img = self.preprocess_image(img)
                    images.append(img)
                    labels.append(item['text'])
        
        return images, labels
    
    def preprocess_image(self, img):
        """Предобработка изображения"""
        # Конвертация в grayscale
        if len(img.shape) == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Ресайз
        img = cv2.resize(img, (self.img_width, self.img_height))
        
        # Нормализация
        img = img.astype(np.float32) / 255.0
        
        # Добавляем канал для совместимости с CNN
        img = np.expand_dims(img, axis=-1)
        
        return img
    
    def build_vocabulary(self, texts):
        """Создание словаря символов"""
        chars = set()
        for text in texts:
            chars.update(text)
        
        # Сортируем символы для воспроизводимости
        chars = sorted(list(chars))
        
        # Добавляем специальные токены
        self.char_to_idx = {char: idx + 1 for idx, char in enumerate(chars)}
        self.char_to_idx[''] = 0  # Пустой символ для CTC
        self.idx_to_char = {idx: char for char, idx in self.char_to_idx.items()}
        self.num_classes = len(self.char_to_idx)
        
        return self.num_classes
    
    def encode_text(self, text):
        """Кодирование текста в последовательность индексов"""
        return [self.char_to_idx.get(char, 0) for char in text]
    
    def decode_text(self, indices):
        """Декодирование последовательности индексов в текст"""
        return ''.join([self.idx_to_char.get(idx, '') for idx in indices if idx != 0])
    
    def prepare_dataset(self, images, labels, batch_size=32, validation_split=0.2):
        """Подготовка TensorFlow Dataset"""
        # Создаем словарь
        all_texts = labels
        self.build_vocabulary(all_texts)
        
        # Кодируем метки
        encoded_labels = [self.encode_text(text) for text in labels]
        
        # Конвертируем в numpy arrays
        images = np.array(images)
        
        # Разделяем на train/val
        X_train, X_val, y_train, y_val = train_test_split(
            images, encoded_labels, test_size=validation_split, random_state=42
        )
        
        # Создаем дат   асеты
        train_dataset = self.create_tf_dataset(X_train, y_train, batch_size, is_training=True)
        val_dataset = self.create_tf_dataset(X_val, y_val, batch_size, is_training=False)
        
        return train_dataset, val_dataset
    
    def create_tf_dataset(self, images, labels, batch_size, is_training=True):
        """Создание TF Dataset с паддингом"""
        # Паддинг меток
        labels_padded = pad_sequences(labels, maxlen=self.max_text_len, padding='post', value=0)
        
        dataset = tf.data.Dataset.from_tensor_slices((images, labels_padded))
        
        if is_training:
            dataset = dataset.shuffle(buffer_size=1000)
            dataset = dataset.batch(batch_size, drop_remainder=True)
            dataset = dataset.map(self.preprocess_batch)
            dataset = dataset.prefetch(tf.data.AUTOTUNE)
        else:
            dataset = dataset.batch(batch_size)
            dataset = dataset.map(self.preprocess_batch)
            dataset = dataset.prefetch(tf.data.AUTOTUNE)
        
        return dataset
    
    def preprocess_batch(self, images, labels):
        """Предобработка батча для обучения"""
        # images уже нормализованы
        return images, labels

# Визуализация данных
def visualize_samples(images, labels, num_samples=5):
    """Визуализация примеров из датасета"""
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 3))
    
    for i in range(num_samples):
        axes[i].imshow(images[i].squeeze(), cmap='gray')
        axes[i].set_title(f'Text: {labels[i]}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

ModuleNotFoundError: No module named 'sklearn'